# VQE del modelo de Ising con campo transverso en GPU

Este notebook prepara 30 VQE para 20 spins: 10 valores de h entre 0.2 y 2.0, con 3 semillas por valor. La ejecución está limitada a un solo caso de verificación para no lanzar todo el barrido automáticamente.

El Hamiltoniano es H = -J sum Z_i Z_(i+1) - h sum X_i, con J=1 y condiciones abiertas. El ansatz tiene 4 bloques de rotaciones RX-RY-RZ en todos los spins y 3 anillos periódicos de CNOTs entre los bloques. El estado se mantiene como un tensor de orden 20 y las contracciones usan PyTorch CUDA.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import torch

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'ising_tfim_gpu.py').exists():
    NOTEBOOK_DIR = Path('Tensor Networks/ising con campo transverso').resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

from ising_tfim_gpu import (
    DEFAULT_FIELDS,
    DEFAULT_SEEDS,
    ENTANGLING_RINGS,
    ROTATION_LAYERS,
    device_report,
    run_grid,
    run_vqe,
    save_results,
    select_device,
)

ROOT = NOTEBOOK_DIR
RESULTS_DIR = ROOT / 'results'
DEVICE = select_device(require_cuda=True)
print(device_report(DEVICE))

## Configuración del problema y del ansatz

DEFAULT_FIELDS contiene los diez valores solicitados y DEFAULT_SEEDS define los tres VQE independientes que se usarán para cada campo.

In [ ]:
NUM_SPINS = 20
COUPLING_J = 1.0
FIRST_FIELD_ADAM_STEPS = 500
LATER_FIELD_ADAM_STEPS = 150
LBFGS_STEPS = 80
LEARNING_RATE = 0.03
OUTPUT_PATH = RESULTS_DIR / 'all_vqe_results_adam500_later150_lbfgs80.json'

print(f'Campos transversales: {list(DEFAULT_FIELDS)}')
print(f'VQE por campo: {len(DEFAULT_SEEDS)}')
print(f'Capas de rotación: {ROTATION_LAYERS}')
print(f'Anillos de CNOT: {ENTANGLING_RINGS}')
print(f'Parámetros entrenables: {ROTATION_LAYERS * NUM_SPINS * 3}')

## Validación: ejecutar únicamente un VQE

Esta celda es la que se debe ejecutar para la comprobación inicial. No llama a run_grid; por tanto, no ejecuta los 30 VQE.

In [ ]:
RUN_ALL = False

if RUN_ALL:
    results = run_grid(
        fields=DEFAULT_FIELDS,
        seeds=DEFAULT_SEEDS,
        num_spins=NUM_SPINS,
        coupling=COUPLING_J,
        first_field_adam_steps=FIRST_FIELD_ADAM_STEPS,
        later_field_adam_steps=LATER_FIELD_ADAM_STEPS,
        lbfgs_steps=LBFGS_STEPS,
        learning_rate=LEARNING_RATE,
        device=DEVICE,
    )
    output_path = save_results(results, OUTPUT_PATH)
    print(f'{len(results)} VQE guardados en {output_path}')
else:
    result = run_vqe(
        num_spins=NUM_SPINS,
        coupling=COUPLING_J,
        field=0.2,
        seed=DEFAULT_SEEDS[0],
        adam_steps=FIRST_FIELD_ADAM_STEPS,
        lbfgs_steps=LBFGS_STEPS,
        learning_rate=LEARNING_RATE,
        device=DEVICE,
    )
    output_path = save_results(result, RESULTS_DIR / 'verification_single_vqe_adam500_lbfgs80.json')
    print(f'VQE individual guardado en {output_path}')
    print(f'Dispositivo: {result["device"]} | GPU: {result["gpu_name"]}')
    print(f'Energía inicial: {result["initial_energy"]:.8f}')
    print(f'Energía final:   {result["final_energy"]:.8f}')
    print(f'Reducción:       {result["energy_drop"]:.8f}')
    print(f'Norma final:      {result["final_state_norm"]:.8f}')
    assert result['device'] == 'cuda'
    assert result['trainable_parameters'] == 240
    assert abs(result['final_state_norm'] - 1.0) < 1e-4
    assert result['final_energy'] < result['initial_energy']
    print('Validación correcta: CUDA, 240 parámetros, norma unitaria y energía reducida.')

In [ ]:
# Gráfica del único VQE ejecutado. Esta celda no lanza optimizaciones nuevas.
if not RUN_ALL:
    plt.figure(figsize=(7, 4))
    plt.plot(result['history'], label='Energía VQE')
    plt.axhline(result['final_energy'], color='tab:red', linestyle='--', label='Energía final')
    plt.xlabel('Paso de optimización')
    plt.ylabel('Energía')
    plt.title('TFIM, 20 spins, h=0.2: Adam + LBFGS')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Barrido completo (no ejecutado en esta validación)

Cuando se quiera lanzar el experimento completo, cambiar RUN_ALL = True en la celda anterior. Se ejecutarán 10 campos x 3 semillas = 30 VQE. Los tres VQE de h=0.2 usarán 500 pasos de Adam y los campos siguientes 150 pasos de Adam con reciclado de parámetros por semilla. Todos tendrán hasta 80 iteraciones de LBFGS y el resultado se guardará en OUTPUT_PATH.